In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Cell 2 — Load and Preprocess Data
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/fabdelja/autism-screening-for-toddlers/Toddler Autism dataset July 2018.csv')

# Drop unnecessary columns
df = df.drop(columns=['Case_No', 'Qchat-10-Score', 'Who completed the test'])

# Encode categorical features
df['Sex'] = (df['Sex'] == 'm').astype(int)
df['Jaundice'] = (df['Jaundice'] == 'yes').astype(int)
df['Family_mem_with_ASD'] = (df['Family_mem_with_ASD'] == 'yes').astype(int)
le = LabelEncoder()
df['Ethnicity'] = le.fit_transform(df['Ethnicity'])
df['target'] = (df['Class/ASD Traits '] == 'Yes').astype(int)
df = df.drop(columns=['Class/ASD Traits '])

# Split
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Ready for training")

X_train: (843, 15)
X_test: (211, 15)
Ready for training


In [3]:
# Cell 3 — Train XGBoost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
xgb_model = XGBClassifier(
    n_estimators = 200, 
    max_depth = 4,
    learning_rate = 0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
print("XGBoost Results:")
print(f"  Accuracy:  {accuracy_score(y_test, xgb_preds):.4f}")
print(f"  F1 Score:  {f1_score(y_test, xgb_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}")

XGBoost Results:
  Accuracy:  0.9763
  F1 Score:  0.9827
  ROC-AUC:   0.9989


In [4]:
# Cell 4 — Train LightGBM

lgbm_model = LGBMClassifier(
    n_estimators = 200,
    max_depth = 4,
    learning_rate = 0.05, 
    class_weights = 'balanced',
    random_state = 42, 
    verbose = -1
    
)

lgbm_model.fit(X_train, y_train)
lgbm_preds = lgbm_model.predict(X_test)

print("LightGBM Results:")
print(f"  Accuracy:  {accuracy_score(y_test, lgbm_preds):.4f}")
print(f"  F1 Score:  {f1_score(y_test, lgbm_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, lgbm_model.predict_proba(X_test)[:,1]):.4f}")

LightGBM Results:
  Accuracy:  0.9716
  F1 Score:  0.9793
  ROC-AUC:   0.9975


In [5]:
# Cell 5 — Cross Validation
xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')

print("XGBoost 5-Fold Cross Validation:")
print(f"  Scores: {xgb_cv.round(4)}")
print(f"  Mean:   {xgb_cv.mean():.4f}")
print(f"  Std:    {xgb_cv.std():.4f}")

XGBoost 5-Fold Cross Validation:
  Scores: [0.9408 0.9645 0.9586 0.9643 0.9524]
  Mean:   0.9561
  Std:    0.0088


In [6]:
# Cell 6 — Save Best Model
joblib.dump(xgb_model, 'asd_xgb_model.pkl')
joblib.dump(le, 'ethnicity_encoder.pkl')

print("Saved:")
print("  asd_xgb_model.pkl — XGBoost classifier")
print("  ethnicity_encoder.pkl — LabelEncoder for ethnicity")
print("\nModule 2 best model: XGBoost")
print(f"  Accuracy:  97.63%")
print(f"  F1 Score:  98.27%")
print(f"  ROC-AUC:   99.89%")
print(f"  CV Mean:   95.61% ± 0.88%")

Saved:
  asd_xgb_model.pkl — XGBoost classifier
  ethnicity_encoder.pkl — LabelEncoder for ethnicity

Module 2 best model: XGBoost
  Accuracy:  97.63%
  F1 Score:  98.27%
  ROC-AUC:   99.89%
  CV Mean:   95.61% ± 0.88%
